In [1]:
from __future__ import annotations

import csv
import signal
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor,
    rationalize,
)

from config.benchmark_config import CEQL_TRAIN, PYSR, SINDY, EQLDIV


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SYMBOLIC_COMPARISON_TIMEOUT_S = 5.0

COEFFICIENT_ROUNDING_ENABLED = True
COEFFICIENT_ROUND_DECIMALS = 2

SCALED_ARGUMENT_CANONICALIZATION_ENABLED = True
SCALED_ARGUMENT_ROUND_DECIMALS = 4
NUMERIC_CONSTANT_ROUND_DECIMALS = 2

NEAR_CONSTANT_COLLAPSE_ENABLED = True
NEAR_CONSTANT_REL_TOL = 1e-4

RATIONAL_CANONICALIZATION_ENABLED = True
RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS = 2

REQUIRE_ALL_REPORTS = True

EQLDIV_ZERO_BASED_VARIABLES = True

MAX_VARIABLES = 10

REPORTS = {
    "ceql": Path(CEQL_TRAIN.results_path),
    "pysr": Path(PYSR.results_path),
    "sindy": Path(SINDY.results_path),
    "eql_div": Path(EQLDIV.results_path),
}


# ------------------------------------------------------------
# Timeout
# ------------------------------------------------------------

class SymbolicComparisonTimeoutError(Exception):
    pass


def _timeout_handler(signum, frame):
    raise SymbolicComparisonTimeoutError


# ------------------------------------------------------------
# SymPy parsing
# ------------------------------------------------------------

TRANSFORMATIONS = standard_transformations + (
    convert_xor,
    implicit_multiplication_application,
    rationalize,
)


def _base_locals() -> dict[str, object]:
    local_dict: dict[str, object] = {
        "sqrt": sp.sqrt,
        "log": sp.log,
        "ln": sp.log,
        "log10": lambda x: sp.log(x, 10),
        "exp": sp.exp,
        "sin": sp.sin,
        "cos": sp.cos,
        "tan": sp.tan,
        "Abs": sp.Abs,
        "abs": sp.Abs,
        "square": lambda x: x**2,
        "cube": lambda x: x**3,
        "pi": sp.pi,
        "E": sp.E,
    }

    for i in range(1, MAX_VARIABLES + 1):
        local_dict[f"x{i}"] = sp.Symbol(f"x{i}")

    local_dict["x"] = sp.Symbol("x1")

    return local_dict


def _found_locals(method: str, expr_str: str) -> dict[str, object]:
    local_dict = _base_locals()

    if method == "eql_div" and EQLDIV_ZERO_BASED_VARIABLES and "x0" in expr_str:
        for i in range(MAX_VARIABLES):
            local_dict[f"x{i}"] = sp.Symbol(f"x{i + 1}")

    return local_dict


def _parse_symbolic_expression(expr_str: str, local_dict: dict[str, object]) -> sp.Expr:
    expr_str = str(expr_str).strip()

    if expr_str == "":
        raise ValueError("empty expression")

    if expr_str.startswith("<symbolic extraction failed"):
        raise ValueError(expr_str)

    expr = parse_expr(
        expr_str,
        local_dict=local_dict,
        transformations=TRANSFORMATIONS,
        evaluate=True,
    )

    if expr.has(sp.nan, sp.oo, -sp.oo, sp.zoo):
        raise ValueError(f"non-finite symbolic expression: {expr}")

    return expr


def parse_target_expression(expr_str: str) -> sp.Expr:
    return _parse_symbolic_expression(
        expr_str=expr_str,
        local_dict=_base_locals(),
    )


def parse_found_expression(expr_str: str, method: str) -> sp.Expr:
    return _parse_symbolic_expression(
        expr_str=expr_str,
        local_dict=_found_locals(method=method, expr_str=expr_str),
    )


# ------------------------------------------------------------
# Symbolic equality
# ------------------------------------------------------------

@dataclass(frozen=True)
class SymbolicMatchResult:
    correct: bool
    successful_check: str
    diff: sp.Expr


def _round_numeric_atoms(expr: sp.Expr, decimals: int) -> sp.Expr:
    replacements = {}

    for atom in expr.atoms(sp.Number):
        if atom in (sp.pi, sp.E, sp.I):
            continue

        if atom.is_Integer:
            continue

        if atom.is_real is False:
            continue

        rounded = round(float(atom), decimals)

        if rounded == 0.0:
            replacements[atom] = sp.Integer(0)
        else:
            replacements[atom] = sp.Rational(str(rounded))

    return expr.xreplace(replacements)


def _round_numeric_constant_expr(expr: sp.Expr, decimals: int) -> sp.Expr:
    value = sp.N(expr, 50)

    if value.is_real is False:
        return expr

    rounded = round(float(value), decimals)

    if rounded == 0.0:
        return sp.Integer(0)

    return sp.Rational(str(rounded))


def _fold_numeric_constant_residual(expr: sp.Expr, decimals: int) -> sp.Expr:
    expr = sp.simplify(expr)

    if len(expr.free_symbols) == 0 and expr.is_number:
        return _round_numeric_constant_expr(expr, decimals)

    return expr


def _round_numeric_multiplicative_coefficients(
    expr: sp.Expr,
    decimals: int,
) -> sp.Expr:
    symbols = sorted(expr.free_symbols, key=lambda s: s.name)

    if not symbols:
        return _round_numeric_constant_expr(expr, decimals)

    expr = sp.expand(expr)
    terms = sp.Add.make_args(expr)
    rounded_terms = []

    for term in terms:
        numeric_part, symbolic_part = term.as_independent(*symbols, as_Add=False)

        if numeric_part.is_number:
            rounded_numeric_part = _round_numeric_constant_expr(
                numeric_part,
                decimals=decimals,
            )
            rounded_terms.append(rounded_numeric_part * symbolic_part)
        else:
            rounded_terms.append(term)

    return sp.expand(sp.Add(*rounded_terms))


def _round_symbolic_residual(diff: sp.Expr, decimals: int) -> sp.Expr:
    diff = sp.expand(diff)

    diff = _round_numeric_multiplicative_coefficients(
        diff,
        decimals=decimals,
    )

    diff = _round_numeric_atoms(diff, decimals)
    diff = sp.expand(diff)
    diff = sp.cancel(sp.together(diff))
    diff = sp.expand(diff)

    return diff


def _round_and_fold_residual(diff: sp.Expr) -> sp.Expr:
    diff = sp.expand(diff)

    diff = _round_numeric_multiplicative_coefficients(
        diff,
        decimals=COEFFICIENT_ROUND_DECIMALS,
    )

    diff = _round_numeric_atoms(
        diff,
        decimals=COEFFICIENT_ROUND_DECIMALS,
    )

    diff = sp.expand(diff)
    diff = sp.cancel(sp.together(diff))
    diff = sp.expand(diff)

    diff = _fold_numeric_constant_residual(
        diff,
        decimals=NUMERIC_CONSTANT_ROUND_DECIMALS,
    )

    diff = sp.expand(diff)
    return diff


def _collapse_near_constant_polynomial_argument(
    arg: sp.Expr,
    rel_tol: float,
) -> sp.Expr:
    symbols = sorted(arg.free_symbols, key=lambda s: s.name)

    if not symbols:
        return arg

    try:
        poly = sp.Poly(sp.expand(arg), *symbols)
    except Exception:
        return arg

    constant_term = poly.coeff_monomial(tuple(0 for _ in symbols))

    nonconstant_coeffs = []

    for monom, coeff in poly.terms():
        if any(power != 0 for power in monom):
            nonconstant_coeffs.append(coeff)

    if not nonconstant_coeffs:
        return constant_term

    if not constant_term.is_number:
        return arg

    numeric_nonconstant_abs = []

    for coeff in nonconstant_coeffs:
        if not coeff.is_number:
            return arg

        numeric_nonconstant_abs.append(abs(float(sp.N(coeff, 50))))

    constant_abs = abs(float(sp.N(constant_term, 50)))

    if constant_abs == 0.0:
        return arg

    max_nonconstant_abs = max(numeric_nonconstant_abs)

    if max_nonconstant_abs / constant_abs <= rel_tol:
        return constant_term

    return arg


def _normalize_scaled_polynomial_argument(
    arg: sp.Expr,
    decimals: int,
) -> tuple[sp.Expr, sp.Expr] | None:
    symbols = sorted(arg.free_symbols, key=lambda s: s.name)

    if not symbols:
        return None

    try:
        expanded_arg = sp.expand(arg)
        poly = sp.Poly(expanded_arg, *symbols)
    except Exception:
        return None

    lc = poly.LC()

    if not lc.is_number:
        return None

    lc_value = float(sp.N(lc, 50))

    if lc_value == 0.0:
        return None

    scale = lc

    normalized_arg = sp.expand(expanded_arg / scale)
    normalized_arg = _round_numeric_atoms(normalized_arg, decimals)
    normalized_arg = sp.expand(normalized_arg)

    return scale, normalized_arg


def _canonicalize_log_part(arg: sp.Expr, decimals: int) -> sp.Expr:
    if NEAR_CONSTANT_COLLAPSE_ENABLED:
        arg = _collapse_near_constant_polynomial_argument(
            arg,
            rel_tol=NEAR_CONSTANT_REL_TOL,
        )

    arg = sp.cancel(sp.together(arg))
    num, den = sp.fraction(arg)

    if den != 1:
        return (
            _canonicalize_log_part(num, decimals=decimals)
            - _canonicalize_log_part(den, decimals=decimals)
        )

    if len(num.free_symbols) == 0:
        return sp.log(num)

    result = _normalize_scaled_polynomial_argument(
        num,
        decimals=decimals,
    )

    if result is None:
        return sp.log(num)

    scale, normalized_arg = result
    return sp.log(scale) + sp.log(normalized_arg)


def _canonicalize_sqrt_part(arg: sp.Expr, decimals: int) -> sp.Expr:
    if NEAR_CONSTANT_COLLAPSE_ENABLED:
        arg = _collapse_near_constant_polynomial_argument(
            arg,
            rel_tol=NEAR_CONSTANT_REL_TOL,
        )

    result = _normalize_scaled_polynomial_argument(
        arg,
        decimals=decimals,
    )

    if result is None:
        return sp.sqrt(arg)

    scale, normalized_arg = result
    return sp.sqrt(scale) * sp.sqrt(normalized_arg)


def _canonicalize_scaled_log_and_sqrt_arguments(
    expr: sp.Expr,
    decimals: int,
) -> sp.Expr:
    def query(e: sp.Expr) -> bool:
        if e.func == sp.log and len(e.args) == 1:
            return True

        if e.is_Pow and e.exp == sp.Rational(1, 2):
            return True

        return False

    def value(e: sp.Expr) -> sp.Expr:
        if e.func == sp.log and len(e.args) == 1:
            return _canonicalize_log_part(
                e.args[0],
                decimals=decimals,
            )

        if e.is_Pow and e.exp == sp.Rational(1, 2):
            return _canonicalize_sqrt_part(
                e.base,
                decimals=decimals,
            )

        return e

    return expr.replace(query, value)


def _normalized_rational_components(expr: sp.Expr) -> tuple[sp.Expr, sp.Expr]:
    expr = sp.cancel(sp.together(expr))

    num, den = sp.fraction(expr)
    symbols = sorted(expr.free_symbols, key=lambda s: s.name)

    if not symbols:
        return sp.expand(num), sp.expand(den)

    try:
        den_poly = sp.Poly(den, *symbols)
    except Exception:
        return sp.expand(num), sp.expand(den)

    lc = den_poly.LC()

    if not lc.is_number:
        return sp.expand(num), sp.expand(den)

    lc_value = float(sp.N(lc, 50))

    if lc_value == 0.0:
        return sp.expand(num), sp.expand(den)

    num = sp.expand(num / lc)
    den = sp.expand(den / lc)

    return num, den


def _rationally_equivalent_after_rounding(
    found: sp.Expr,
    target: sp.Expr,
) -> SymbolicMatchResult:
    found_num, found_den = _normalized_rational_components(found)
    target_num, target_den = _normalized_rational_components(target)

    cross_residual = sp.expand(found_num * target_den - target_num * found_den)

    cross_residual = _round_numeric_multiplicative_coefficients(
        cross_residual,
        decimals=RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS,
    )

    cross_residual = _round_numeric_atoms(
        cross_residual,
        decimals=RATIONAL_CROSS_RESIDUAL_ROUND_DECIMALS,
    )

    cross_residual = sp.expand(cross_residual)
    cross_residual = sp.factor(cross_residual)
    cross_residual = sp.expand(cross_residual)

    rational_checks: list[tuple[str, Callable[[sp.Expr], sp.Expr]]] = [
        ("rational_cross_direct", lambda z: z),
        ("rational_cross_expand", sp.expand),
        ("rational_cross_factor", sp.factor),
        ("rational_cross_simplify", sp.simplify),
    ]

    for name, check in rational_checks:
        checked = check(cross_residual)

        if checked == 0:
            return SymbolicMatchResult(
                correct=True,
                successful_check=name,
                diff=sp.Integer(0),
            )

    return SymbolicMatchResult(
        correct=False,
        successful_check="",
        diff=cross_residual,
    )


def symbolic_equivalent_no_timeout(found: sp.Expr, target: sp.Expr) -> SymbolicMatchResult:
    diff = found - target

    exact_checks: list[tuple[str, Callable[[sp.Expr], sp.Expr]]] = [
        ("direct", lambda z: z),
        ("expand", sp.expand),
        ("cancel_together", lambda z: sp.cancel(sp.together(z))),
        ("factor", sp.factor),
        ("simplify", sp.simplify),
    ]

    for name, check in exact_checks:
        checked = check(diff)

        if checked == 0:
            return SymbolicMatchResult(
                correct=True,
                successful_check=name,
                diff=sp.Integer(0),
            )

    if COEFFICIENT_ROUNDING_ENABLED:
        rounded_diff = _round_symbolic_residual(
            diff,
            decimals=COEFFICIENT_ROUND_DECIMALS,
        )

        rounded_diff = _fold_numeric_constant_residual(
            rounded_diff,
            decimals=NUMERIC_CONSTANT_ROUND_DECIMALS,
        )

        rounded_checks: list[tuple[str, Callable[[sp.Expr], sp.Expr]]] = [
            ("rounded_direct", lambda z: z),
            ("rounded_expand", sp.expand),
            ("rounded_cancel_together", lambda z: sp.cancel(sp.together(z))),
            ("rounded_factor", sp.factor),
            ("rounded_simplify", sp.simplify),
        ]

        for name, check in rounded_checks:
            checked = check(rounded_diff)

            if checked == 0:
                return SymbolicMatchResult(
                    correct=True,
                    successful_check=name,
                    diff=sp.Integer(0),
                )

    if SCALED_ARGUMENT_CANONICALIZATION_ENABLED:
        canonical_found = _canonicalize_scaled_log_and_sqrt_arguments(
            found,
            decimals=SCALED_ARGUMENT_ROUND_DECIMALS,
        )

        canonical_target = _canonicalize_scaled_log_and_sqrt_arguments(
            target,
            decimals=SCALED_ARGUMENT_ROUND_DECIMALS,
        )

        canonical_diff = canonical_found - canonical_target
        canonical_diff = _round_and_fold_residual(canonical_diff)

        canonical_checks: list[tuple[str, Callable[[sp.Expr], sp.Expr]]] = [
            ("scaled_arg_direct", lambda z: z),
            ("scaled_arg_expand", sp.expand),
            ("scaled_arg_cancel_together", lambda z: sp.cancel(sp.together(z))),
            ("scaled_arg_factor", sp.factor),
            ("scaled_arg_simplify", sp.simplify),
        ]

        for name, check in canonical_checks:
            checked = check(canonical_diff)

            if checked == 0:
                return SymbolicMatchResult(
                    correct=True,
                    successful_check=name,
                    diff=sp.Integer(0),
                )

    if RATIONAL_CANONICALIZATION_ENABLED:
        rational_match = _rationally_equivalent_after_rounding(
            found=found,
            target=target,
        )

        if rational_match.correct:
            return rational_match

    return SymbolicMatchResult(
        correct=False,
        successful_check="",
        diff=sp.cancel(sp.together(diff)),
    )


def symbolic_equivalent(found: sp.Expr, target: sp.Expr) -> SymbolicMatchResult:
    if SYMBOLIC_COMPARISON_TIMEOUT_S is None or SYMBOLIC_COMPARISON_TIMEOUT_S <= 0:
        return symbolic_equivalent_no_timeout(found, target)

    old_handler = signal.getsignal(signal.SIGALRM)

    try:
        signal.signal(signal.SIGALRM, _timeout_handler)
        signal.setitimer(signal.ITIMER_REAL, SYMBOLIC_COMPARISON_TIMEOUT_S)

        result = symbolic_equivalent_no_timeout(found, target)

        signal.setitimer(signal.ITIMER_REAL, 0.0)
        return result

    except SymbolicComparisonTimeoutError:
        return SymbolicMatchResult(
            correct=False,
            successful_check="timeout",
            diff=sp.Symbol(
                f"SYMBOLIC_COMPARISON_TIMEOUT_AFTER_{SYMBOLIC_COMPARISON_TIMEOUT_S}_SECONDS"
            ),
        )

    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old_handler)


# ------------------------------------------------------------
# Report collection
# ------------------------------------------------------------

@dataclass(frozen=True)
class RowResult:
    method: str
    group: str
    run: str
    seed: str
    correct: bool
    successful_check: str
    true_expr: str
    found_expr: str
    error: str


def evaluate_report(method: str, path: Path) -> list[RowResult]:
    if not path.exists():
        message = f"Missing report for method '{method}': {path}"
        if REQUIRE_ALL_REPORTS:
            raise FileNotFoundError(message)
        print(message)
        return []

    results: list[RowResult] = []

    with path.open("r", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            group = row.get("group", "")
            run = row.get("run", "")
            seed = row.get("seed", "")

            true_expr_str = row.get("true_expr", "")
            found_expr_str = row.get("found_expr", "")

            try:
                target = parse_target_expression(true_expr_str)
                found = parse_found_expression(
                    expr_str=found_expr_str,
                    method=method,
                )

                match = symbolic_equivalent(found=found, target=target)

                error = ""
                if match.successful_check == "timeout":
                    error = f"symbolic comparison timeout after {SYMBOLIC_COMPARISON_TIMEOUT_S} s"

                results.append(
                    RowResult(
                        method=method,
                        group=group,
                        run=run,
                        seed=seed,
                        correct=match.correct,
                        successful_check=match.successful_check,
                        true_expr=true_expr_str,
                        found_expr=found_expr_str,
                        error=error,
                    )
                )

            except Exception as e:
                results.append(
                    RowResult(
                        method=method,
                        group=group,
                        run=run,
                        seed=seed,
                        correct=False,
                        successful_check="",
                        true_expr=true_expr_str,
                        found_expr=found_expr_str,
                        error=str(e),
                    )
                )

    return results


# ------------------------------------------------------------
# Printing
# ------------------------------------------------------------

def print_per_run_results(results: list[RowResult]) -> None:
    print("\nPer-run symbolic match results")
    print("=" * 80)

    for r in results:
        status = "CORRECT" if r.correct else "WRONG"

        print(
            f"{r.method:8s} | {r.group:8s} | run={r.run:>3s} | "
            f"seed={r.seed:>5s} | {status}"
        )

        if r.correct:
            print(f"  check: {r.successful_check}")
        else:
            print(f"  true : {r.true_expr}")
            print(f"  found: {r.found_expr}")
            if r.error:
                print(f"  error: {r.error}")

        print("-" * 80)


def print_recovery_rate_by_method_and_equation(results: list[RowResult]) -> None:
    print("\nRecovery rate by method and equation")
    print("=" * 80)

    keys = sorted({(r.method, r.group) for r in results})

    print(f"{'method':8s} | {'equation':45s} | recovery_rate")
    print("-" * 80)

    for method, group in keys:
        sub = [r for r in results if r.method == method and r.group == group]
        n_total = len(sub)
        n_correct = sum(r.correct for r in sub)

        recovery_rate = n_correct / n_total if n_total else 0.0

        print(f"{method:8s} | {group:45s} | {recovery_rate:.6f}")


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main() -> None:
    all_results: list[RowResult] = []

    for method, path in REPORTS.items():
        method_results = evaluate_report(method=method, path=path)
        all_results.extend(method_results)

    print_per_run_results(all_results)
    print_recovery_rate_by_method_and_equation(all_results)


if __name__ == "__main__":
    main()


Per-run symbolic match results
ceql     | expr_000_lin_uni | run=  0 | seed=    0 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_000_lin_uni | run=  1 | seed=    1 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_000_lin_uni | run=  2 | seed=    2 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_000_lin_uni | run=  3 | seed=    3 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_000_lin_uni | run=  4 | seed=    4 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_001_lin_bi | run=  0 | seed=    0 | CORRECT
  check: direct
--------------------------------------------------------------------------------
ceql     | expr_001_l